# Entregable 1 - Fundamentacion en Analitica Estrategica de Datos

Este notebook resuelve el PD completo sobre preprocesamiento de un modelo estrella con:
- tabla de hechos: FACT_VENTAS_MASTER.csv
- dimension tiendas: DIM_TIENDAS.csv
- dimension productos: DIM_PRODUCTOS.csv

Incluye: carga, exploracion, limpieza, transformacion, validacion de integridad referencial, integracion, analisis (15 consultas) y generacion de `ventas_limpias.csv`.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

## 1) Carga de los datos

In [2]:
fact_ventas = pd.read_csv("FACT_VENTAS_MASTER.csv")
dim_tienda = pd.read_csv("DIM_TIENDAS.csv")
dim_producto = pd.read_csv("DIM_PRODUCTOS.csv")

print("fact_ventas:", fact_ventas.shape)
print("dim_tienda:", dim_tienda.shape)
print("dim_producto:", dim_producto.shape)

fact_ventas: (10000, 23)
dim_tienda: (20, 3)
dim_producto: (150, 4)


## 2) Exploracion inicial

In [3]:
def perfil_tabla(df, nombre):
    resumen = pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "nulos_%": (df.isna().mean() * 100).round(2),
        "unicos": df.nunique(dropna=True)
    })
    print(f"\n==== {nombre} ====")
    print("filas:", len(df), " | columnas:", df.shape[1])
    print("duplicados exactos:", df.duplicated().sum())
    display(df.head(3))
    display(resumen)

perfil_tabla(fact_ventas, "fact_ventas")
perfil_tabla(dim_tienda, "dim_tienda")
perfil_tabla(dim_producto, "dim_producto")


==== fact_ventas ====
filas: 10000  | columnas: 23
duplicados exactos: 0


,Venta_ID,Fecha_ISO,Tienda_ID,SKU_ID,Pais_Venta,CLIENTE_NOMBRE_PII,CLIENTE_EMAIL_PII,CLIENTE_EDAD_PII,Moneda_Transaccion,Monto_Total_Local,Impuesto_Local,Tasa_Referencia_Dia,Metodo_Pago,Es_Online,Canal_Origen,Costo_Envio_USD,Latitud_Tienda,Longitud_Tienda,Browser_User_Agent,ID_Session_Web,Flag_Fraude,Version_Esquema,Ref_S3_TXT
0,TXN-600000,2025-08-11T15:00:20.491637,STORE-001,SKU-00077,UK,Null User,user_0@uk.com,44.00,GBP,"1,024.76",146.39,0.79,Crypto,No,App,34.67,-18.01,0.08,Mozilla/5.0... (Simulado),0xcb2c83f6,0,v2.4.1,TXN-600000.txt
1,TXN-600001,2026-01-29T15:00:20.511040,STORE-017,SKU-00008,USA,Müller S.,user_1@usa.com,33.00,USD,706.16,35.31,1.00,Crypto,No,App,45.37,-38.37,127.37,Mozilla/5.0... (Simulado),0xd141323f,0,v2.4.1,TXN-600001.txt
2,TXN-600002,2025-10-30T15:00:20.526154,STORE-009,SKU-00082,USA,Müller S.,user_2@usa.com,25.00,USD,"1,394.57",69.73,1.00,Crypto,True,Web,25.97,15.90,-47.23,Mozilla/5.0... (Simulado),0x74f10ced,0,v2.4.1,TXN-600002.txt


,tipo,nulos,nulos_%,unicos
Venta_ID,object,0,0.00,10000
Fecha_ISO,object,0,0.00,10000
Tienda_ID,object,0,0.00,20
SKU_ID,object,0,0.00,631
Pais_Venta,object,0,0.00,5
CLIENTE_NOMBRE_PII,object,0,0.00,5
CLIENTE_EMAIL_PII,object,0,0.00,10000
CLIENTE_EDAD_PII,float64,1004,10.04,62
Moneda_Transaccion,object,0,0.00,6
Monto_Total_Local,float64,0,0.00,755



==== dim_tienda ====
filas: 20  | columnas: 3
duplicados exactos: 0


,Tienda_ID,Nombre_Tienda,Pais
0,STORE-001,GlobalTech UK - 1,UK
1,STORE-002,GlobalTech USA - 2,USA
2,STORE-003,GlobalTech Colombia - 3,Colombia


,tipo,nulos,nulos_%,unicos
Tienda_ID,object,0,0.00,20
Nombre_Tienda,object,0,0.00,20
Pais,object,0,0.00,5



==== dim_producto ====
filas: 150  | columnas: 4
duplicados exactos: 0


,SKU_ID,Nombre_Producto,Categoria,Costo_Base_USD
0,SKU-00001,Gadget Pro X1,Oficina,"1,040.91"
1,SKU-00002,Gadget Pro X2,Electr0nica,381.08
2,SKU-00003,Gadget Pro X3,Hogar,"1,044.33"


,tipo,nulos,nulos_%,unicos
SKU_ID,object,0,0.00,150
Nombre_Producto,object,0,0.00,150
Categoria,object,0,0.00,9
Costo_Base_USD,float64,0,0.00,150


## 3) Limpieza de datos

In [4]:
# Copias para limpieza
fact = fact_ventas.copy()
tienda = dim_tienda.copy()
producto = dim_producto.copy()

# Eliminar duplicados exactos
fact = fact.drop_duplicates().copy()
tienda = tienda.drop_duplicates().copy()
producto = producto.drop_duplicates().copy()

# Estandarizar espacios en texto
for col in ["Venta_ID", "Tienda_ID", "SKU_ID", "Pais_Venta", "Moneda_Transaccion", "Metodo_Pago", "Es_Online", "Canal_Origen"]:
    fact[col] = fact[col].astype(str).str.strip()

for col in ["Tienda_ID", "Nombre_Tienda", "Pais"]:
    tienda[col] = tienda[col].astype(str).str.strip()

for col in ["SKU_ID", "Nombre_Producto", "Categoria"]:
    producto[col] = producto[col].astype(str).str.strip()

# Correccion de categorias inconsistentes
map_categoria = {
    "Electr0nica": "Electronica",
    "Electrónica": "Electronica",
    "Ofic_Ina": "Oficina",
    "Gaminng": "Gaming",
    "G@ming": "Gaming",
    "H0gar": "Hogar"
}
producto["Categoria"] = producto["Categoria"].replace(map_categoria)

print("Categorias depuradas:", sorted(producto["Categoria"].unique()))

Categorias depuradas: ['Electronica', 'Gaming', 'Hogar', 'Oficina']


## 4) Manejo de nulos y conversion de tipos

In [5]:
# Fechas
fact["Fecha_ISO"] = pd.to_datetime(fact["Fecha_ISO"], errors="coerce")

# Numericos
fact["Monto_Total_Local"] = pd.to_numeric(fact["Monto_Total_Local"], errors="coerce")
fact["CLIENTE_EDAD_PII"] = pd.to_numeric(fact["CLIENTE_EDAD_PII"], errors="coerce")

# Nulos de edad -> mediana
edad_mediana = fact["CLIENTE_EDAD_PII"].median()
fact["CLIENTE_EDAD_PII"] = fact["CLIENTE_EDAD_PII"].fillna(edad_mediana)

# Booleano inconsistente en Es_Online
map_online = {
    "true": 1, "1": 1, "si": 1, "sí": 1, "yes": 1,
    "false": 0, "0": 0, "no": 0
}
fact["Es_Online"] = (
    fact["Es_Online"]
    .astype(str)
    .str.lower()
    .str.strip()
    .replace(map_online)
)
fact["Es_Online"] = pd.to_numeric(fact["Es_Online"], errors="coerce").fillna(0).astype(int)

# Metodo de pago
fact["Metodo_Pago"] = fact["Metodo_Pago"].replace({"N/A": "No_Especificado", "Unknown": "No_Especificado"})

# Consistencia moneda vs pais de venta
map_moneda = {
    "UK": "GBP",
    "USA": "USD",
    "México": "MXN",
    "Colombia": "COP",
    "España": "EUR"
}
fact["Moneda_Esperada"] = fact["Pais_Venta"].map(map_moneda)
mask_moneda = fact["Moneda_Esperada"].notna() & (fact["Moneda_Transaccion"] != fact["Moneda_Esperada"])
fact.loc[mask_moneda, "Moneda_Transaccion"] = fact.loc[mask_moneda, "Moneda_Esperada"]

print("Nulos restantes en Fecha_ISO:", fact["Fecha_ISO"].isna().sum())
print("Nulos restantes en Monto_Total_Local:", fact["Monto_Total_Local"].isna().sum())
print("Nulos restantes en CLIENTE_EDAD_PII:", fact["CLIENTE_EDAD_PII"].isna().sum())

Nulos restantes en Fecha_ISO: 0
Nulos restantes en Monto_Total_Local: 0
Nulos restantes en CLIENTE_EDAD_PII: 0


C:\Users\Diego Reyes\AppData\Local\Temp\ipykernel_15116\3262361890.py:22: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(map_online)


## 5) Estandarizar nombres de columnas

In [6]:
fact = fact.rename(columns={
    "Tienda_ID": "id_tienda",
    "SKU_ID": "id_producto",
    "Fecha_ISO": "fecha",
    "Monto_Total_Local": "precio",
    "Pais_Venta": "pais_venta",
    "Moneda_Transaccion": "moneda_transaccion",
    "Metodo_Pago": "metodo_pago",
    "Es_Online": "es_online"
})

tienda = tienda.rename(columns={
    "Tienda_ID": "id_tienda",
    "Nombre_Tienda": "nombre_tienda",
    "Pais": "pais_tienda"
})

producto = producto.rename(columns={
    "SKU_ID": "id_producto",
    "Nombre_Producto": "nombre_producto",
    "Categoria": "categoria",
    "Costo_Base_USD": "costo_base_usd"
})

fact.head(2)

,Venta_ID,fecha,id_tienda,id_producto,pais_venta,CLIENTE_NOMBRE_PII,CLIENTE_EMAIL_PII,CLIENTE_EDAD_PII,moneda_transaccion,precio,Impuesto_Local,Tasa_Referencia_Dia,metodo_pago,es_online,Canal_Origen,Costo_Envio_USD,Latitud_Tienda,Longitud_Tienda,Browser_User_Agent,ID_Session_Web,Flag_Fraude,Version_Esquema,Ref_S3_TXT,Moneda_Esperada
0,TXN-600000,2025-08-11 15:00:20.491637,STORE-001,SKU-00077,UK,Null User,user_0@uk.com,44.00,GBP,"1,024.76",146.39,0.79,Crypto,0,App,34.67,-18.01,0.08,Mozilla/5.0... (Simulado),0xcb2c83f6,0,v2.4.1,TXN-600000.txt,GBP
1,TXN-600001,2026-01-29 15:00:20.511040,STORE-017,SKU-00008,USA,Müller S.,user_1@usa.com,33.00,USD,706.16,35.31,1.00,Crypto,0,App,45.37,-38.37,127.37,Mozilla/5.0... (Simulado),0xd141323f,0,v2.4.1,TXN-600001.txt,USD


## 6) Validacion de integridad referencial

In [7]:
ids_tienda_invalidos = sorted(set(fact["id_tienda"]) - set(tienda["id_tienda"]))
ids_producto_invalidos = sorted(set(fact["id_producto"]) - set(producto["id_producto"]))

filas_invalidas_tienda = (~fact["id_tienda"].isin(tienda["id_tienda"])).sum()
filas_invalidas_producto = (~fact["id_producto"].isin(producto["id_producto"])).sum()

print("IDs tienda invalidos (unicos):", len(ids_tienda_invalidos))
print("Filas con id_tienda invalido:", filas_invalidas_tienda)
print("IDs producto invalidos (unicos):", len(ids_producto_invalidos))
print("Filas con id_producto invalido:", filas_invalidas_producto)

fact_valida = fact[
    fact["id_tienda"].isin(tienda["id_tienda"]) &
    fact["id_producto"].isin(producto["id_producto"])
].copy()

print("Filas fact originales:", len(fact))
print("Filas fact validas para integrar:", len(fact_valida))

IDs tienda invalidos (unicos): 0
Filas con id_tienda invalido: 0
IDs producto invalidos (unicos): 481
Filas con id_producto invalido: 487
Filas fact originales: 10000
Filas fact validas para integrar: 9513


## 7) Creacion de columnas calculadas

In [8]:
fact_valida["cantidad"] = 1
fact_valida["total_venta"] = fact_valida["precio"] * fact_valida["cantidad"]
fact_valida["anio"] = fact_valida["fecha"].dt.year
fact_valida["mes"] = fact_valida["fecha"].dt.month
fact_valida["trimestre"] = fact_valida["fecha"].dt.to_period("Q").astype(str)

fact_valida[["fecha", "cantidad", "precio", "total_venta", "anio", "mes", "trimestre"]].head(3)

,fecha,cantidad,precio,total_venta,anio,mes,trimestre
0,2025-08-11 15:00:20.491637,1,"1,024.76","1,024.76",2025,8,2025Q3
1,2026-01-29 15:00:20.511040,1,706.16,706.16,2026,1,2026Q1
2,2025-10-30 15:00:20.526154,1,"1,394.57","1,394.57",2025,10,2025Q4


## 8) Integracion del modelo estrella

In [9]:
ventas_limpias = (
    fact_valida
    .merge(producto, on="id_producto", how="left")
    .merge(tienda, on="id_tienda", how="left")
)

print("ventas_limpias:", ventas_limpias.shape)
display(ventas_limpias.head(3))

ventas_limpias: (9513, 34)


,Venta_ID,fecha,id_tienda,id_producto,pais_venta,CLIENTE_NOMBRE_PII,CLIENTE_EMAIL_PII,CLIENTE_EDAD_PII,moneda_transaccion,precio,Impuesto_Local,Tasa_Referencia_Dia,metodo_pago,es_online,Canal_Origen,Costo_Envio_USD,Latitud_Tienda,Longitud_Tienda,Browser_User_Agent,ID_Session_Web,Flag_Fraude,Version_Esquema,Ref_S3_TXT,Moneda_Esperada,cantidad,total_venta,anio,mes,trimestre,nombre_producto,categoria,costo_base_usd,nombre_tienda,pais_tienda
0,TXN-600000,2025-08-11 15:00:20.491637,STORE-001,SKU-00077,UK,Null User,user_0@uk.com,44.00,GBP,"1,024.76",146.39,0.79,Crypto,0,App,34.67,-18.01,0.08,Mozilla/5.0... (Simulado),0xcb2c83f6,0,v2.4.1,TXN-600000.txt,GBP,1,"1,024.76",2025,8,2025Q3,Gadget Pro X77,Oficina,926.55,GlobalTech UK - 1,UK
1,TXN-600001,2026-01-29 15:00:20.511040,STORE-017,SKU-00008,USA,Müller S.,user_1@usa.com,33.00,USD,706.16,35.31,1.00,Crypto,0,App,45.37,-38.37,127.37,Mozilla/5.0... (Simulado),0xd141323f,0,v2.4.1,TXN-600001.txt,USD,1,706.16,2026,1,2026Q1,Gadget Pro X8,Hogar,504.40,GlobalTech USA - 17,USA
2,TXN-600002,2025-10-30 15:00:20.526154,STORE-009,SKU-00082,USA,Müller S.,user_2@usa.com,25.00,USD,"1,394.57",69.73,1.00,Crypto,1,Web,25.97,15.90,-47.23,Mozilla/5.0... (Simulado),0x74f10ced,0,v2.4.1,TXN-600002.txt,USD,1,"1,394.57",2025,10,2025Q4,Gadget Pro X82,Electronica,996.12,GlobalTech USA - 9,USA


## 9) Archivo de salida: ventas_limpias.csv

In [10]:
columnas_salida = [
    "Venta_ID", "fecha", "id_tienda", "nombre_tienda", "pais_tienda",
    "id_producto", "nombre_producto", "categoria", "costo_base_usd",
    "cantidad", "precio", "total_venta", "moneda_transaccion", "metodo_pago",
    "es_online", "anio", "mes", "trimestre"
]

ventas_limpias[columnas_salida].to_csv("ventas_limpias.csv", index=False)
print("Archivo generado: ventas_limpias.csv")

Archivo generado: ventas_limpias.csv


## 10) Analisis de resultados (15 consultas)

In [11]:
# 1) Categoria que genera mayores ingresos
q1 = ventas_limpias.groupby("categoria")["total_venta"].sum().sort_values(ascending=False)

# 2) Cantidad total de unidades vendidas por producto
q2 = ventas_limpias.groupby(["id_producto", "nombre_producto"], as_index=False)["cantidad"].sum().sort_values("cantidad", ascending=False)

# 3) Ingreso total generado por cada producto
q3 = ventas_limpias.groupby(["id_producto", "nombre_producto"], as_index=False)["total_venta"].sum().sort_values("total_venta", ascending=False)

# 4) Promedio de ventas por tienda
q4 = ventas_limpias.groupby(["id_tienda", "nombre_tienda"], as_index=False)["total_venta"].mean().sort_values("total_venta", ascending=False)

# 5) Total de ventas por pais
q5 = ventas_limpias.groupby("pais_tienda", as_index=False)["total_venta"].sum().sort_values("total_venta", ascending=False)

# 6) Pais con mayores ingresos
q6 = q5.head(1)

# 7) Total de ingresos por trimestre
q7 = ventas_limpias.groupby("trimestre", as_index=False)["total_venta"].sum().sort_values("trimestre")

# 8) Mes con mayor volumen de ventas (unidades)
q8 = ventas_limpias.groupby("mes", as_index=False)["cantidad"].sum().sort_values("cantidad", ascending=False)

# 9) Desempeno de tiendas por categoria
q9 = pd.pivot_table(
    ventas_limpias,
    index=["id_tienda", "nombre_tienda"],
    columns="categoria",
    values="total_venta",
    aggfunc="sum",
    fill_value=0
).sort_index()

# 10) Top 5 productos con mayores ingresos
q10 = q3.head(5)

# 11) Top 5 tiendas con mayor cantidad de unidades vendidas
q11 = ventas_limpias.groupby(["id_tienda", "nombre_tienda"], as_index=False)["cantidad"].sum().sort_values("cantidad", ascending=False).head(5)

# 12) Evolucion mensual de ventas por categoria
ventas_limpias["periodo_mensual"] = ventas_limpias["fecha"].dt.to_period("M").astype(str)
q12 = ventas_limpias.groupby(["periodo_mensual", "categoria"], as_index=False)["total_venta"].sum().sort_values(["periodo_mensual", "total_venta"], ascending=[True, False])

# 13) Producto con mayor ingreso dentro de cada categoria
q13_base = ventas_limpias.groupby(["categoria", "id_producto", "nombre_producto"], as_index=False)["total_venta"].sum()
q13 = q13_base.loc[q13_base.groupby("categoria")["total_venta"].idxmax()].sort_values("total_venta", ascending=False)

# 14) Tiendas con bajo desempeno en ventas (<= percentil 25)
q14_base = ventas_limpias.groupby(["id_tienda", "nombre_tienda"], as_index=False)["total_venta"].sum()
umbral_bajo = q14_base["total_venta"].quantile(0.25)
q14 = q14_base[q14_base["total_venta"] <= umbral_bajo].sort_values("total_venta")

# 15) Promedio de precio por categoria (costo base USD)
q15 = ventas_limpias.groupby("categoria", as_index=False)["costo_base_usd"].mean().sort_values("costo_base_usd", ascending=False)

print("1) Categoria con mayores ingresos:")
display(q1.head(1))

print("2) Unidades vendidas por producto (top 10):")
display(q2.head(10))

print("3) Ingreso total por producto (top 10):")
display(q3.head(10))

print("4) Promedio de ventas por tienda:")
display(q4)

print("5) Total de ventas por pais:")
display(q5)

print("6) Pais con mayores ingresos:")
display(q6)

print("7) Total de ingresos por trimestre:")
display(q7)

print("8) Mes con mayor volumen de ventas:")
display(q8.head(1))

print("9) Desempeno de tiendas por categoria (matriz):")
display(q9)

print("10) Top 5 productos con mayores ingresos:")
display(q10)

print("11) Top 5 tiendas con mayor cantidad de unidades vendidas:")
display(q11)

print("12) Evolucion mensual de ventas por categoria (primeras 20 filas):")
display(q12.head(20))

print("13) Producto con mayor ingreso dentro de cada categoria:")
display(q13)

print("14) Tiendas con bajo desempeno en ventas:")
display(q14)

print("15) Promedio de precio por categoria (costo base USD):")
display(q15)

1) Categoria con mayores ingresos:


categoria
Oficina   1,571,806,565.77
Name: total_venta, dtype: float64

2) Unidades vendidas por producto (top 10):


,id_producto,nombre_producto,cantidad
121,SKU-00122,Gadget Pro X122,91
131,SKU-00132,Gadget Pro X132,90
62,SKU-00063,Gadget Pro X63,89
32,SKU-00033,Gadget Pro X33,85
10,SKU-00011,Gadget Pro X11,78
17,SKU-00018,Gadget Pro X18,78
136,SKU-00137,Gadget Pro X137,77
114,SKU-00115,Gadget Pro X115,76
116,SKU-00117,Gadget Pro X117,76
147,SKU-00148,Gadget Pro X148,75


3) Ingreso total por producto (top 10):


...

,categoria,id_producto,nombre_producto,total_venta
132,Oficina,SKU-00073,Gadget Pro X73,"112,457,035.50"
99,Hogar,SKU-00098,Gadget Pro X98,"90,977,586.71"
47,Gaming,SKU-00057,Gadget Pro X57,"90,831,301.39"
12,Electronica,SKU-00051,Gadget Pro X51,"59,965,666.12"


14) Tiendas con bajo desempeno en ventas:


,id_tienda,nombre_tienda,total_venta
0,STORE-001,GlobalTech UK - 1,"317,020.60"
5,STORE-006,GlobalTech UK - 6,"319,470.66"
7,STORE-008,GlobalTech UK - 8,"323,288.72"
14,STORE-015,GlobalTech España - 15,"325,860.81"
6,STORE-007,GlobalTech España - 7,"354,409.16"


15) Promedio de precio por categoria (costo base USD):


,categoria,costo_base_usd
3,Oficina,701.21
1,Gaming,637.42
2,Hogar,573.21
0,Electronica,485.76


## 11) Hallazgos de preprocesamiento

In [12]:
hallazgos = {
    "registros_fact_originales": int(len(fact)),
    "registros_fact_integrables": int(len(fact_valida)),
    "registros_excluidos_por_integridad": int(len(fact) - len(fact_valida)),
    "ids_producto_invalidos_unicos": int(len(ids_producto_invalidos)),
    "ids_tienda_invalidos_unicos": int(len(ids_tienda_invalidos)),
    "categorias_finales": sorted(producto["categoria"].unique().tolist()),
    "pais_top_ingresos": str(q6.iloc[0]["pais_tienda"]),
    "categoria_top_ingresos": str(q1.index[0])
}

for k, v in hallazgos.items():
    print(f"{k}: {v}")

registros_fact_originales: 10000
registros_fact_integrables: 9513
registros_excluidos_por_integridad: 487
ids_producto_invalidos_unicos: 481
ids_tienda_invalidos_unicos: 0
categorias_finales: ['Electronica', 'Gaming', 'Hogar', 'Oficina']
pais_top_ingresos: Colombia
categoria_top_ingresos: Oficina


## 12) Conclusiones

- Se construyo un flujo completo de preprocesamiento para el modelo estrella.
- Se corrigieron inconsistencias de categorias, monedas, booleanos y valores faltantes clave.
- Se validaron llaves foraneas y se excluyeron registros no integrables.
- Se genero el archivo final `ventas_limpias.csv` listo para analisis adicional o visualizacion.